### PyObject vs PyVarObject. Хранение в памяти int, list; None, True, False

In [8]:
import sys

# PyObject: ob_refcnt + ob_type (фиксированный размер)
# PyVarObject: ob_refcnt + ob_type + ob_size (переменный размер)

# int — PyVarObject (длинная арифметика, ob_size = кол-во "цифр" по 30 бит)
print(sys.getsizeof(0))       # 28 байт (ob_size=0)
print(sys.getsizeof(1))       # 28 байт (ob_size=1)
print(sys.getsizeof(2**30))   # 32 байта (ob_size=2)
print(sys.getsizeof(2**60))   # 36 байт (ob_size=3)


24
28
32
36


In [ ]:

# list — PyVarObject (массив указателей PyObject*)
print(sys.getsizeof([]))          # пустой list
print(sys.getsizeof([1, 2, 3]))   # 3 указателя


In [11]:

# None, True, False — синглтоны
a = None
b = None
print(a is b)  # True
print(id(a), id(b))

print(True is True)    # True
print(False is False)  # True

True
4401714640 4401714640
True
True


In [ ]:
# interning малых int (-5..256)
x = 256
y = 256
print(x is y)  # True

x = 257
y = 257
print(x is y)  # False (вне интерактивного режима; зависит от реализации)


True
False
4401714640 4401712296 4401712392
34854


In [14]:
print(id(None), id(True), id(False))

import ctypes
# ob_refcnt у None
print(sys.getrefcount(None))

4401714640 4401712296 4401712392
34848


### `__defaults__`: дефолтные значения у функций

In [16]:
def append_to(element, target=[]):
    target.append(element)
    return target

print(append_to(1))  # [1]
print(append_to(2))  # [1, 2] — мутабельный дефолт расшаривается!

# дефолты хранятся в атрибуте __defaults__
print(append_to.__defaults__)  # ([1, 2],)

# можно изменить извне
append_to.__defaults__ = ([],)
print(append_to(42))  # [42]

# правильный паттерн
def append_safe(element, target=None):
    if target is None:
        target = []
    target.append(element)
    return target


[1]
[1, 2]
([1, 2],)
[42]


### Хранение объектов и классов в памяти Python

In [18]:
class MyClass:
    class_var = 10
    def __init__(self, x):
        self.x = x

    def some_method(self):
        ...

obj = MyClass(42)

# класс хранит свои атрибуты в __dict__
print(MyClass.__dict__.keys())  # class_var, __init__, __dict__, ...

# объект хранит instance-атрибуты в своём __dict__
print(obj.__dict__)  # {'x': 42}

# class_var не в obj.__dict__, но доступен через цепочку поиска
print(obj.class_var)        # 10
print('class_var' in obj.__dict__)  # False

# __class__ — ссылка на класс
print(obj.__class__ is MyClass)  # True
print(type(MyClass))             # <class 'type'>

obj.some_method()  # объект obj типа MyClass хранит в себе ссылку на объект MyClass типа type. Идёт по ней в MyClass.some_method и передаёт себя в поле self

dict_keys(['__module__', 'class_var', '__init__', 'some_method', '__dict__', '__weakref__', '__doc__'])
{'x': 42}
10
False
True
<class 'type'>


### `__init__` vs `__new__`

In [19]:
class Foo:
    def __new__(cls, val):
        print(f"__new__ called, cls={cls}")
        instance = super().__new__(cls)
        return instance

    def __init__(self, val):
        print(f"__init__ called, self={self}")
        self.val = val

obj = Foo(10)
# __new__ создаёт объект (аллокация), __init__ инициализирует (заполняет атрибуты)

# __new__ может вернуть экземпляр другого класса — тогда __init__ НЕ вызовется
class Bar:
    def __new__(cls):
        return 42  # не экземпляр Bar
    def __init__(self):
        print("never called")

b = Bar()
print(b, type(b))  # 42 <class 'int'>


__new__ called, cls=<class '__main__.Foo'>
__init__ called, self=<__main__.Foo object at 0x107c675b0>
42 <class 'int'>


### Реализация singleton через `__new__`

In [20]:
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value=None):
        self.value = value

a = Singleton(1)
b = Singleton(2)
print(a is b)    # True
print(a.value)   # 2 — __init__ вызывается каждый раз


True
2


### Реализация singleton через декоратор

In [ ]:
def singleton(cls):
    instances = {}  # get_instance замыкается на instances, поэтому он не очистится после выхода из singleton
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance

@singleton
class Database:
    def __init__(self, url="localhost"):
        self.url = url

a = Database("server1")
b = Database("server2")
print(a is b)   # True
print(a.url)    # server1 — второй вызов проигнорирован


True
server1


### Копирование: `=` vs `copy` vs `copy.deepcopy`

In [22]:
import copy

original = [[1, 2], [3, 4]]

# = : просто ещё одна ссылка
assigned = original
assigned[0][0] = 999
print(original)  # [[999, 2], [3, 4]]

original = [[1, 2], [3, 4]]

# copy.copy: новый внешний список, но внутренние — те же объекты
shallow = copy.copy(original)
shallow[0][0] = 888
print(original)  # [[888, 2], [3, 4]] — внутренний список общий
shallow.append([5])
print(len(original))  # 2 — внешний список разный

original = [[1, 2], [3, 4]]

# copy.deepcopy: рекурсивное копирование
deep = copy.deepcopy(original)
deep[0][0] = 777
print(original)  # [[1, 2], [3, 4]] — полностью независим


[[999, 2], [3, 4]]
[[888, 2], [3, 4]]
2
[[1, 2], [3, 4]]


### `__del__`. Вызов. Обработка ошибок. Воскрешение

In [26]:
import gc

class Destructor:
    def __del__(self):
        print(f"__del__ called for {id(self)}")

obj = Destructor()
del obj  # уменьшает refcount; __del__ вызывается когда refcount == 0

# исключения в __del__ подавляются (только warning в stderr)
class BadDel:
    def __del__(self):
        raise ValueError("oops")

b = BadDel()
del b  # Exception ignored in __del__

# воскрешение объекта
revived = None

class Zombie:
    def __del__(self):
        global revived
        revived = self  # снова создаём ссылку — объект "воскресает"
        print("resurrected!")

z = Zombie()
del z
gc.collect()

print(revived)  # <__main__.Zombie object ...>


Exception ignored in: <function BadDel.__del__ at 0x107a59120>
Traceback (most recent call last):
  File "/var/folders/vp/0v99fjwx2nlcysq5ns40bc_w0000gn/T/ipykernel_99751/3895051725.py", line 13, in __del__
ValueError: oops


__del__ called for 4424635136
resurrected!


### Наследование. __mro__ vs __bases__

In [27]:
class A: pass
class B(A): pass
class C(A): pass
class D(B, C): pass

# __bases__ — непосредственные родители
print(D.__bases__)  # (<class 'B'>, <class 'C'>)
print(B.__bases__)  # (<class 'A'>,)

# __mro__ — полный порядок разрешения методов (включая себя и object)
print(D.__mro__)  # (D, B, C, A, object)

(<class '__main__.B'>, <class '__main__.C'>)
(<class '__main__.A'>,)
(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)


### Method Resolution Order. C3-линеаризация. Контрпример: не BFS и не DFS

In [29]:
#     X
#    / \
#   Y   Z
#   |   |
#   M   N
#    \ /
#     K

class X:
    def who(self): return "X"
class Y(X): pass
class Z(X):
    def who(self): return "Z"
class M(Y): pass
class N(Z): pass
class K(M, N): pass

print(K.__mro__)
# K -> M -> Y -> N -> Z -> X -> object
# BFS дал бы: K -> M -> N -> Y -> Z -> X -> object (Y и N поменяны с M/N)
# DFS дал бы: K -> M -> Y -> X -> N -> Z -> X -> object (X слишком рано)


(<class '__main__.K'>, <class '__main__.M'>, <class '__main__.Y'>, <class '__main__.N'>, <class '__main__.Z'>, <class '__main__.X'>, <class 'object'>)


### Property: @property, @setter

In [33]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        print(f'Setting temperature to {value}')
        if value < -273.15:
            raise ValueError("Below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9/5 + 32

t = Temperature(100)
print(t.celsius)      # 100
print(t.fahrenheit)   # 212.0
t.celsius = -10
print(t.celsius)      # -10
# t.celsius = -300    # ValueError
# t.fahrenheit = 50   # AttributeError — нет setter


100
212.0
Setting temperature to -10
-10


### Динамическая работа с атрибутами: getattr, setattr, delattr

In [34]:
class Box:
    def __init__(self):
        self.items = []

b = Box()

setattr(b, 'color', 'red')
print(b.color)               # red
print(getattr(b, 'color'))   # red
print(getattr(b, 'weight', 0))  # 0 — дефолт, если нет атрибута

delattr(b, 'color')
print(hasattr(b, 'color'))   # False

# Полезно для динамического доступа
fields = ['x', 'y', 'z']
for i, name in enumerate(fields):
    setattr(b, name, i)
print(b.__dict__)  # {'items': [], 'x': 0, 'y': 1, 'z': 2}


red
red
0
False
{'items': [], 'x': 0, 'y': 1, 'z': 2}


### @dataclass; @total_ordering

In [35]:
from dataclasses import dataclass
from functools import total_ordering

@dataclass
class Point:
    x: float
    y: float

p = Point(1, 2)
print(p)            # Point(x=1, y=2) — автоматический __repr__
print(p == Point(1, 2))  # True — автоматический __eq__

# @total_ordering: определяем __eq__ и один оператор сравнения — остальные генерируются
@total_ordering
class Student:
    def __init__(self, name, grade):
        self.name = name
        self.grade = grade
    def __eq__(self, other):
        return self.grade == other.grade
    def __lt__(self, other):
        return self.grade < other.grade

a = Student("Alice", 90)
b = Student("Bob", 85)
print(a > b)   # True — сгенерировано из __lt__ + __eq__
print(a >= b)   # True
print(a <= b)   # False


Point(x=1, y=2)
True
True
True
False


### @dataclass: `field`, `__post_init__`

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Config:
    name: str
    tags: list = field(default_factory=list)  # мутабельный дефолт — через field
    _hash: int = field(init=False, repr=False)  # не в __init__, не в __repr__

    def __post_init__(self):
        self._hash = hash(self.name)
        self.name = self.name.strip().upper()

c1 = Config("  hello  ")
c2 = Config("world")
print(c1)       # Config(name='HELLO', tags=[])
print(c1._hash) # hash('  hello  ') — вычислен в __post_init__
c1.tags.append("a")
print(c2.tags)  # [] — у каждого свой список благодаря default_factory


### raise ... from

In [ ]:
# raise X from Y — явная цепочка исключений, устанавливает __cause__
def convert(value):
    try:
        return int(value)
    except ValueError as e:
        raise TypeError(f"Cannot convert {value!r}") from e

try:
    convert("abc")
except TypeError as e:
    print(e)            # Cannot convert 'abc'
    print(e.__cause__)  # invalid literal for int()...

# raise X from None — подавляет неявный контекст
def clean_convert(value):
    try:
        return int(value)
    except ValueError:
        raise TypeError(f"Bad value: {value!r}") from None

try:
    clean_convert("abc")
except TypeError as e:
    print(e.__cause__)    # None


# convert("abc")  # напечатает 2 исключения
# clean_convert("abc")  # только TypeError

Cannot convert 'abc'
invalid literal for int() with base 10: 'abc'
None
invalid literal for int() with base 10: 'abc'


### Name mangling

In [6]:
class BankAccount:
    def __init__(self, balance):
        self._balance = balance  # "защищённый"
        self.__pin = "1234"      # name mangling
    
    def get_balance(self):
        return self._balance
    
    def __verify_pin(self, pin):
        return self.__pin == pin  # внутри класса доступно

# Доступ:
acc = BankAccount(100)
print(acc._balance)        # 100 (но так не принято)
# print(acc.__pin)         # AttributeError!
print(acc._BankAccount__pin)  # "1234" (но лучше так не делать)

100
1234
